# Hindi Text Restoration and Translation - End-to-End (Colab)

This notebook trains and runs the complete assignment pipeline. No LLM is
used at any stage.

| Stage | Model | Task |
| --- | --- | --- |
| 1 | `bert-base-multilingual-cased` + LoRA | restore commas and sentence-final marks (`। ? !`) |
| 2 | `bert-base-multilingual-cased` + LoRA | pairwise sentence ordering + TF-IDF clustering |
| 3 | `ai4bharat/indictrans2-indic-en-dist-200M` | Hindi -> English translation |

All models are loaded one at a time and released before the next stage.
A hard 1.5 GB CUDA memory cap is applied, matching the assignment limit.

**Runtime > Change runtime type > Hardware accelerator: GPU (T4).**

## 0. Project location

Upload the project folder to Colab (or clone your GitHub fork) and set
`PROJECT_DIR` below. The folder must contain `src/`, `data/raw/`,
`requirements-colab.txt` and `scripts/colab_setup.sh`.

In [ ]:
PROJECT_DIR = "/content/nlp-zero-to-hero"

import os
import sys

assert os.path.isdir(PROJECT_DIR), f"Upload/clone the project to {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Files:", sorted(os.listdir("."))[:12])

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("Total GPU memory (GB):", round(props.total_memory / 1024**3, 2))

## 1. Install dependencies

This installs `transformers`, `peft`, `datasets`, `sacrebleu` and the
IndicTrans2 preprocessing toolkit. It takes a few minutes on a fresh Colab
runtime. PyTorch is already provided by Colab and is not reinstalled.

In [ ]:
!bash scripts/colab_setup.sh

## 2. Imports, seeds and the memory cap

`limit_gpu_memory(1.5)` calls
`torch.cuda.set_per_process_memory_fraction(1.5 / total_memory)`, so the
process is killed instead of silently exceeding the budget.

In [ ]:
import json
import time
from pathlib import Path

from src.constants import (
    GROUP_SIZE,
    MEMORY_LIMIT_GB,
    PUNCT_DATA,
    ORDER_DATA,
    PUNCT_MODEL_DIR,
    ORDER_MODEL_DIR,
    TRANSLATION_MODEL,
    SEED,
)
from src.utils import gpu_memory, limit_gpu_memory, read_jsonl, set_seed

set_seed(SEED)
print("Memory cap:", limit_gpu_memory(MEMORY_LIMIT_GB))
print("Memory now:", gpu_memory())

## 3. Build the datasets from Hindi Wikipedia

The download is streamed, so only the requested articles are kept. Each
article is split with `str.split()` (never a `\w` regex, which would break
Devanagari matras) and split into train/validation/test by article.

* **Punctuation windows:** 32-96 words, sentences shuffled inside each
  window. Only the first subword of each word is supervised.
* **Ordering pairs:** consecutive sentences, plus the swapped pair as a
  negative example.

For a faster first run use `--num-articles 1200`.

In [ ]:
!python -m src.data_prep --num-articles 3000 --max-train-pairs 40000

In [ ]:
summary = json.loads(Path("data/processed/data_prep_summary.json").read_text())
print("Articles:", summary["articles"])
print("Splits:", summary["splits"])
print("Punctuation examples:")
for split, stats in summary["punctuation"].items():
    labels = ", ".join(f"{k}={v['count']}" for k, v in stats["labels"].items())
    print(f"  {split}: {stats['examples']} windows | {labels}")
print("Ordering pairs:", {k: v["pairs"] for k, v in summary["ordering"].items()})

example = read_jsonl(PUNCT_DATA / "train.jsonl")[0]
print("\nSample words :", example["words"][:18])
print("Sample labels:", example["labels"][:18])

## 4. Stage 1 - punctuation restoration (BERT + LoRA)

* Labels: `none`, `comma`, `sentence_end`, `question`, `exclamation`.
* LoRA `r=16`, `alpha=32`, targets `query`/`value`, classification head
  trained fully.
* Class-weighted loss helps the rare `question` and `exclamation` classes.
* fp16 + gradient checkpointing keep the run inside the 1.5 GB budget.

In [ ]:
!python -m src.punctuation.train \
    --epochs 3 \
    --batch-size 8 \
    --learning-rate 3e-4 \
    --memory-gb 1.5

In [ ]:
punct_summary = json.loads(
    (PUNCT_MODEL_DIR / "training_summary.json").read_text()
)
print("Trainable parameters:", punct_summary.get("trainable_parameters", "see log above"))
print("Validation:", json.dumps(punct_summary["validation_metrics"], indent=2))
print("Test:", json.dumps(punct_summary["test_metrics"], indent=2))
print("Peak memory:", punct_summary["memory_at_end"])

## 5. Stage 2 - sentence ordering (mBERT pairwise + LoRA)

The classifier sees `[CLS] A [SEP] B [SEP]` and predicts whether A comes
before B. At inference time:

* **Borda (default):** average the probability that a sentence precedes all
  others; sorting by that score is a robust Kemeny/Borda approximation.
* **clusters:** TF-IDF (char 2-4 grams) + KMeans, order cluster medoids,
  then beam-search inside each cluster.
* **beam:** beam-search a chain over all sentences.

All three decoders are compared in the evaluation below.

In [ ]:
!python -m src.ordering.train \
    --epochs 2 \
    --batch-size 8 \
    --learning-rate 2e-4 \
    --memory-gb 1.5

In [ ]:
order_summary = json.loads(
    (ORDER_MODEL_DIR / "training_summary.json").read_text()
)
print("Validation:", json.dumps(order_summary["validation_metrics"], indent=2))
print("Test:", json.dumps(order_summary["test_metrics"], indent=2))
print("Peak memory:", order_summary["memory_at_end"])

## 6. Evaluate all three stages

* **Punctuation:** token accuracy, boundary precision/recall/F1, per-class F1.
* **Ordering:** pair accuracy and Kendall's tau on shuffled held-out
  articles, for every decoder plus a TF-IDF-only ablation.
* **Translation:** corpus BLEU and chrF on 200 IITB hi-en test sentences
  (skipped gracefully if the dataset is unavailable).

In [ ]:
!python -m src.evaluate \
    --max-punct-examples 1200 \
    --max-order-pairs 3000 \
    --max-order-articles 30 \
    --max-translation-examples 200

In [ ]:
evaluation = json.loads(Path("reports/evaluation.json").read_text())
print(json.dumps(evaluation, indent=2, ensure_ascii=False)[:6000])

## 7. Run the full pipeline on the assignment files

This writes four files per input: `*.sentences_hi.txt`, `*.groups_hi.txt`,
`*.sentences_en.txt`, `*.groups_en.txt`. Every group contains five
consecutive sentences.

In [ ]:
!python -m src.pipeline --strategy borda --memory-gb 1.5

In [ ]:
for name in ["sample-1.groups_hi.txt", "sample-1.groups_en.txt"]:
    path = Path("outputs", name)
    print("=" * 100)
    print(name, f"({len(path.read_text(encoding='utf-8').splitlines())} groups)")
    for index, line in enumerate(path.read_text(encoding="utf-8").splitlines()[:4]):
        print(f"[group {index + 1}] {line}")

## 8. Memory and timing report

The pipeline releases each model before loading the next one, so the
1.5 GB cap applies to a single stage, never the sum of all three.

In [ ]:
run_summary = json.loads(Path("reports/run_summary.json").read_text())
print("Ordering strategy:", run_summary["ordering_strategy"])
print("Timings (seconds):", json.dumps(run_summary["timings_seconds"], indent=2))
print("Memory limit (GB):", run_summary["memory_limit_gb"])
for stem, stats in run_summary["files"].items():
    print(stem, stats)

## 9. Save the results

The zip contains the Hindi/English outputs and the JSON reports. The LoRA
adapters are small too, but the notebook can always retrain them.

In [ ]:
import shutil

shutil.make_archive("/content/hindi_assignment_outputs", "zip", "outputs")
shutil.copytree("reports", "/content/hindi_assignment_reports", dirs_exist_ok=True)
print("Saved /content/hindi_assignment_outputs.zip")
print("Saved /content/hindi_assignment_reports/")

## 10. Notes and limitations

* **No LLM anywhere.** The only models are mBERT and IndicTrans2.
* **No fallback translation model.** If IndicTrans2 cannot be loaded, the
  pipeline raises an error instead of switching to a different model.
* **Memory.** Every stage is capped at 1.5 GB. Training uses fp16 and
  gradient checkpointing; if you still hit OOM, reduce `--batch-size` to 4
  or `--max-train-samples`.
* **Data.** Wikipedia provides sentence boundaries and punctuation, but no
  paragraph-order labels are needed because pairs are made from consecutive
  sentences.
* **Question/exclamation marks are rare in Wikipedia.** Their F1 can be low;
  the class-weighted loss mitigates but does not remove this.
* **Ordering is hard on real text.** Topical clusters do not always match
  the original narrative order, which is why the notebook compares all
  decoders. Pick the best-performing strategy for the final run.
* **Translation evaluation** uses IITB hi-en test sentences and is skipped
  when the dataset cannot be downloaded.

### Troubleshooting

* `IndicTransToolkit` build fails: run
  `bash scripts/colab_setup.sh` again, or install it from source as the
  script does.
* `peft`/`transformers` mismatch: `pip install --upgrade -r requirements-colab.txt` (keep transformers 4.44.x).
* CUDA OOM during training: lower the batch size, keep gradient
  accumulation at 2, and reload the runtime.